## Confronto tra i V0 e i BRT

In [ ]:
from pathlib import Path
from functools import lru_cache

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter


# ============================================================
# CONFIGURAZIONE
# ============================================================

METRICS = (
    "euclidean, dce, ttc, eggert, time_eggert, rss, sff, "
    "euc_dce_from_v0, euc_dce_minimum, "
    "eggert_ttc_from_v0, eggert_ttc_minimum"
)

# Valori disponibili nei selettori
V_H_VALUES = [3.0, 4.5, 6.0, 7.5, 9.0]       # m/s
V_E_VALUES = [3.0, 4.5, 6.0, 7.5, 9.0]       # m/s
DELTA_E_VALUES = [-0.1, -0.05, 0.0, 0.05, 0.1]  # rad

DEFAULT_METRICS = ["euclidean", "dce"]

SMOOTH_SIGMA = 1.0
OPACITY = 0.5

COLORS = [
    "#1f77b4",
    "#d62728",
    "#2ca02c",
    "#9467bd",
    "#ff7f0e",
    "#17becf",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
]


# ============================================================
# CARICAMENTO METRICHE
# ============================================================

metrics = list(dict.fromkeys(
    name.strip().lower()
    for name in METRICS.split(",")
    if name.strip()
))

root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "results" / "brt").is_dir()
    ),
    None,
)

if root is None:
    raise FileNotFoundError("Cartella results/brt non trovata.")

paths = {}
grids = {}

for metric in metrics:

    path = root / "results" / "brt" / f"{metric}.npz"

    if not path.is_file():
        raise FileNotFoundError(f"File non trovato: {path}")

    with np.load(path, allow_pickle=False) as data:

        if "target_time" in data.files:
            if not np.isclose(float(data["target_time"]), -3.0):
                raise ValueError(
                    f"{metric}: il BRT salvato non è a t = -3 s."
                )

        grids[metric] = {
            axis: np.asarray(data[axis], dtype=float)
            for axis in [
                "x_rel", "y_rel", "theta_rel",
                "v_H", "delta_E", "v_E"
            ]
        }

    paths[metric] = path


# ============================================================
# INTERPOLAZIONE
# ============================================================

def interpolation_weights(axis, value):

    if value < axis[0] or value > axis[-1]:
        raise ValueError(
            f"Valore {value:g} fuori dalla griglia "
            f"[{axis[0]:g}, {axis[-1]:g}]"
        )

    i = int(np.searchsorted(axis, value))

    if i == 0:
        return [(0, 1.0)]

    if i == len(axis):
        return [(len(axis) - 1, 1.0)]

    i0, i1 = i - 1, i

    weight = (value - axis[i0]) / (axis[i1] - axis[i0])

    return [
        (i0, 1.0 - weight),
        (i1, weight),
    ]


@lru_cache(maxsize=32)
def get_slice(metric, field, v_h, delta_e, v_e):

    grid = grids[metric]

    w_h = interpolation_weights(grid["v_H"], v_h)
    w_d = interpolation_weights(grid["delta_E"], delta_e)
    w_e = interpolation_weights(grid["v_E"], v_e)

    with np.load(paths[metric], allow_pickle=False) as data:

        values_6d = data[field]

        values_3d = np.zeros(
            values_6d.shape[:3],
            dtype=float,
        )

        for ih, wh in w_h:
            for idelta, wd in w_d:
                for ie, we in w_e:

                    values_3d += (
                        wh * wd * we
                        * values_6d[:, :, :, ih, idelta, ie]
                    )

    if SMOOTH_SIGMA > 0:
        values_3d = gaussian_filter(
            values_3d,
            sigma=SMOOTH_SIGMA,
            mode=("nearest", "nearest", "wrap"),
        )

    return values_3d.ravel()


@lru_cache(maxsize=None)
def get_coordinates(metric):

    grid = grids[metric]

    X, Y, THETA = np.meshgrid(
        grid["x_rel"],
        grid["y_rel"],
        grid["theta_rel"],
        indexing="ij",
    )

    return X.ravel(), Y.ravel(), THETA.ravel()


# ============================================================
# SELETTORI
# ============================================================

def make_dropdown(values, unit, description, default):

    options = [
        (f"{float(value):g} {unit}", float(value))
        for value in values
    ]

    selected = min(
        options,
        key=lambda option: abs(option[1] - default),
    )[1]

    return widgets.Dropdown(
        options=options,
        value=selected,
        description=description,
        style={"description_width": "50px"},
        layout=widgets.Layout(width="210px"),
    )


field_selector = widgets.Dropdown(
    options=[
        ("V₀", "V0"),
        ("V(−3) / BRT", "BRT"),
    ],
    value="V0",
    description="Campo:",
    style={"description_width": "55px"},
    layout=widgets.Layout(width="220px"),
)

v_h_selector = make_dropdown(
    V_H_VALUES, "m/s", "v_H:", 6.0
)

v_e_selector = make_dropdown(
    V_E_VALUES, "m/s", "v_E:", 6.0
)

delta_selector = make_dropdown(
    DELTA_E_VALUES, "rad", "δ_E:", 0.0
)


# ============================================================
# MENU METRICHE CON SELEZIONE MULTIPLA
# ============================================================

metric_checks = {
    metric: widgets.Checkbox(
        value=(metric in DEFAULT_METRICS),
        description=metric,
        indent=False,
        layout=widgets.Layout(width="240px"),
    )
    for metric in metrics
}

metric_button = widgets.ToggleButton(
    value=False,
    description="Metriche ▾",
    layout=widgets.Layout(width="210px"),
)

metric_menu = widgets.VBox(
    list(metric_checks.values()),
    layout=widgets.Layout(
        display="none",
        border="1px solid #cccccc",
        padding="8px",
        width="245px",
        max_height="300px",
        overflow_y="auto",
    ),
)


def toggle_metric_menu(change):

    metric_menu.layout.display = (
        "flex" if change["new"] else "none"
    )

    metric_button.description = (
        "Metriche ▴" if change["new"] else "Metriche ▾"
    )


metric_button.observe(
    toggle_metric_menu,
    names="value",
)


# ============================================================
# VISUALIZZAZIONE 3D
# ============================================================

plot_output = widgets.Output()


def update_plot(change=None):

    selected_metrics = [
        metric
        for metric, checkbox in metric_checks.items()
        if checkbox.value
    ]

    with plot_output:

        clear_output(wait=True)

        if not selected_metrics:
            print("Seleziona almeno una metrica.")
            return

        field = field_selector.value

        v_h = v_h_selector.value
        v_e = v_e_selector.value
        delta_e = delta_selector.value

        fig = go.Figure()

        warnings = []

        for metric in selected_metrics:

            try:
                values = get_slice(
                    metric,
                    field,
                    v_h,
                    delta_e,
                    v_e,
                )

            except ValueError as exc:
                warnings.append(f"{metric}: {exc}")
                continue

            if (
                not np.isfinite(values).all()
                or np.min(values) > 0
                or np.max(values) < 0
            ):
                warnings.append(
                    f"{metric}: nessuna superficie V=0 "
                    "nella selezione corrente."
                )
                continue

            X, Y, THETA = get_coordinates(metric)

            color = COLORS[metrics.index(metric) % len(COLORS)]

            fig.add_trace(
                go.Isosurface(
                    x=X,
                    y=Y,
                    z=THETA,
                    value=values,
                    isomin=0.0,
                    isomax=0.0,
                    surface_count=1,
                    colorscale=[
                        [0, color],
                        [1, color],
                    ],
                    opacity=OPACITY,
                    showscale=False,
                    showlegend=True,
                    name=metric,
                    hoverinfo="skip",
                    caps=dict(
                        x_show=False,
                        y_show=False,
                        z_show=False,
                    ),
                    lighting=dict(
                        ambient=0.65,
                        diffuse=0.8,
                        specular=0.15,
                        roughness=0.8,
                        fresnel=0.1,
                    ),
                    lightposition=dict(
                        x=100,
                        y=150,
                        z=200,
                    ),
                )
            )

        title = (
            "V₀ = 0"
            if field == "V0"
            else "V(−3) = 0"
        )

        fig.update_layout(
            title=dict(
                text=(
                    f"{title} | "
                    f"v_H = {v_h:g} m/s, "
                    f"v_E = {v_e:g} m/s, "
                    f"δ_E = {delta_e:g} rad"
                ),
                x=0.5,
                xanchor="center",
            ),
            template="plotly_white",
            height=750,
            margin=dict(
                l=10, r=10, b=10, t=75
            ),
            showlegend=True,
            legend=dict(
                x=1.0,
                y=1.0,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(255,255,255,0.7)",
            ),
            scene=dict(
                xaxis_title="x_rel [m]",
                yaxis_title="y_rel [m]",
                zaxis_title="θ_rel [rad]",
                aspectmode="manual",
                aspectratio=dict(
                    x=1.5,
                    y=1.0,
                    z=0.8,
                ),
                camera=dict(
                    eye=dict(
                        x=1.55,
                        y=1.55,
                        z=1.15,
                    )
                ),
            ),
        )

        fig.show(
            config={
                "displayModeBar": False,
                "responsive": True,
            }
        )

        for warning in warnings:
            print(warning)


# ============================================================
# COLLEGAMENTO CONTROLLI
# ============================================================

for selector in [
    field_selector,
    v_h_selector,
    v_e_selector,
    delta_selector,
]:
    selector.observe(update_plot, names="value")

for checkbox in metric_checks.values():
    checkbox.observe(update_plot, names="value")


# ============================================================
# INTERFACCIA
# ============================================================

controls = widgets.HBox(
    [
        widgets.VBox([
            field_selector,
            metric_button,
            metric_menu,
        ]),
        widgets.VBox([
            v_h_selector,
            v_e_selector,
            delta_selector,
        ]),
    ],
    layout=widgets.Layout(
        align_items="flex-start",
        justify_content="flex-start",
    ),
)

display(controls, plot_output)

update_plot()

Output()

# Confronto slice

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# CONFIGURAZIONE
# ============================================================

METRICS = "euclidean, dce, ttc, eggert, time_eggert, rss, sff, euc_dce_from_v0, euc_dce_minimum, eggert_ttc_from_v0, eggert_ttc_minimum"

EGO_SPEEDS = [3.0, 6.0, 9.0]       # Colonne: v_E [m/s]
HUMAN_SPEEDS = [3.0, 6.0, 9.0]     # Righe: v_H [m/s]

THETA_REL = 0.0                    # rad
DELTA_E = 0.0                      # rad

# Le nove fette condividono SEMPRE una sola colorbar.
# True: scala comune anche tra metriche, separatamente per V0 e BRT.
# False: scala adattata alla metrica selezionata.
SCALA_COMUNE_TRA_METRICHE = False

COLORSCALE = "RdYlGn"

EGO_LENGTH = 4.68                  # m
EGO_WIDTH = 2.20                   # m

# ============================================================
# CONTROLLI E CARTELLA RISULTATI
# ============================================================

metrics = list(dict.fromkeys(
    name.strip().lower()
    for name in METRICS.split(",")
    if name.strip()
))



if len(EGO_SPEEDS) != 3 or len(HUMAN_SPEEDS) != 3:
    raise ValueError("Servono esattamente tre velocità ego e tre human.")

root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "results" / "brt").is_dir()
    ),
    None,
)

if root is None:
    raise FileNotFoundError("Cartella results/brt non trovata.")


def nearest(axis, value):
    return int(np.argmin(np.abs(axis - value)))


def color_limits(values):
    finite = np.asarray(values)
    finite = finite[np.isfinite(finite)]

    if finite.size == 0:
        raise ValueError("Una selezione non contiene valori finiti.")

    low, high = float(finite.min()), float(finite.max())

    if low == high:
        padding = max(abs(low), 1.0) * 1e-6
        low, high = low - padding, high + padding

    return low, high


# ============================================================
# CARICAMENTO DELLE NOVE FETTE PER METRICA
# ============================================================

loaded = {}
reference_axes = None

for metric in metrics:
    path = root / "results" / "brt" / f"{metric}.npz"

    with np.load(path, allow_pickle=False) as data:
        axes = {
            name: np.asarray(data[name], dtype=float)
            for name in [
                "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E"
            ]
        }

        # Stessa griglia per confrontare le stesse configurazioni.
        if reference_axes is None:
            reference_axes = axes
        else:
            for name, axis in axes.items():
                reference = reference_axes[name]

                if (
                    axis.shape != reference.shape
                    or not np.allclose(axis, reference)
                ):
                    raise ValueError(
                        f"{metric}: griglia diversa lungo {name}."
                    )

        if "target_time" in data.files:
            if not np.isclose(float(data["target_time"]), -3.0):
                raise ValueError(f"{metric}: il BRT non è a t = -3 s.")

        # Selezione del nodo più vicino, periodica per theta_rel.
        theta_distance = np.abs(np.angle(
            np.exp(1j * (axes["theta_rel"] - THETA_REL))
        ))

        i_theta = int(np.argmin(theta_distance))
        i_delta = nearest(axes["delta_E"], DELTA_E)
        i_human = [nearest(axes["v_H"], v) for v in HUMAN_SPEEDS]
        i_ego = [nearest(axes["v_E"], v) for v in EGO_SPEEDS]

        if len(set(i_human)) != 3 or len(set(i_ego)) != 3:
            raise ValueError(
                "Due velocità selezionano lo stesso nodo: "
                "modifica le velocità."
            )

        loaded[metric] = {}

        for field in ("V0", "BRT"):
            volume = data[field]

            loaded[metric][field] = [
                np.array(
                    volume[:, :, i_theta, ih, i_delta, ie].T,
                    dtype=float,
                    copy=True,
                )
                for ih in i_human
                for ie in i_ego
            ]

            del volume

x = reference_axes["x_rel"]
y = reference_axes["y_rel"]

human_used = reference_axes["v_H"][i_human]
ego_used = reference_axes["v_E"][i_ego]

# ============================================================
# LIMITI DELLA COLORBAR
# ============================================================

scale_limits = {}

for field in ("V0", "BRT"):
    metric_limits = {
        metric: [
            color_limits(z)
            for z in loaded[metric][field]
        ]
        for metric in metrics
    }

    if SCALA_COMUNE_TRA_METRICHE:
        low = min(
            bounds[0]
            for bounds_list in metric_limits.values()
            for bounds in bounds_list
        )

        high = max(
            bounds[1]
            for bounds_list in metric_limits.values()
            for bounds in bounds_list
        )

        for metric in metrics:
            scale_limits[(metric, field)] = (low, high)

    else:
        for metric, bounds_list in metric_limits.items():
            scale_limits[(metric, field)] = (
                min(bounds[0] for bounds in bounds_list),
                max(bounds[1] for bounds in bounds_list),
            )

# ============================================================
# FIGURA E DATI DEI SELETTORI
# ============================================================

fig = make_subplots(
    rows=3,
    cols=3,
    horizontal_spacing=0.055,
    vertical_spacing=0.09,
    subplot_titles=[
        f"v_H = {vh:.2f} m/s · v_E = {ve:.2f} m/s"
        for vh in human_used
        for ve in ego_used
    ],
)

updates = {
    field: {"z": [], "zmin": [], "zmax": []}
    for field in ("V0", "BRT")
}

trace_metrics = []

for metric in metrics:
    for panel in range(9):
        row, col = divmod(panel, 3)
        row, col = row + 1, col + 1

        initial_z = loaded[metric]["V0"][panel]
        low, high = scale_limits[(metric, "V0")]

        # Mappa: interpolazione grafica per colori più continui.
        fig.add_trace(
            go.Heatmap(
                x=x,
                y=y,
                z=initial_z,
                zmin=low,
                zmax=high,
                zsmooth="best",
                colorscale=COLORSCALE,
                showscale=(panel == 2),
                colorbar=dict(
                    x=1.02,
                    y=0.5,
                    len=1.0,
                    thickness=15,
                    tickformat=".2g",
                ),
                hoverinfo="skip",
                visible=(metric == metrics[0]),
            ),
            row=row,
            col=col,
        )

        # Bordo zero calcolato sui dati originali.
        fig.add_trace(
            go.Contour(
                x=x,
                y=y,
                z=initial_z,
                zmin=low,
                zmax=high,
                autocontour=False,
                contours=dict(
                    start=0,
                    end=0,
                    size=1,
                    coloring="none",
                    showlabels=False,
                ),
                line=dict(color="black", width=2.5),
                showscale=False,
                showlegend=False,
                hoverinfo="skip",
                visible=(metric == metrics[0]),
            ),
            row=row,
            col=col,
        )

        trace_metrics.extend([metric, metric])

        for field in ("V0", "BRT"):
            low, high = scale_limits[(metric, field)]

            # Aggiorna sia la mappa sia il contorno.
            updates[field]["z"].extend(
                [loaded[metric][field][panel]] * 2
            )
            updates[field]["zmin"].extend([low, low])
            updates[field]["zmax"].extend([high, high])

# ============================================================
# BORDO EGO E ASSI
# ============================================================

for panel in range(9):
    row, col = divmod(panel, 3)
    row, col = row + 1, col + 1

    axis_id = "" if panel == 0 else str(panel + 1)

    # Rettangolo ego con bordo nero su fondo bianco.
    # L'interno resta trasparente.
    for color, width in [("white", 5), ("black", 2)]:
        fig.add_shape(
            type="rect",
            x0=-EGO_LENGTH / 2,
            x1=EGO_LENGTH / 2,
            y0=-EGO_WIDTH / 2,
            y1=EGO_WIDTH / 2,
            line=dict(color=color, width=width),
            fillcolor="rgba(0,0,0,0)",
            layer="above",
            row=row,
            col=col,
        )

    fig.update_xaxes(
        title_text="x_rel [m]" if row == 3 else None,
        range=[float(x.min()), float(x.max())],
        constrain="domain",
        showgrid=False,
        zeroline=False,
        row=row,
        col=col,
    )

    fig.update_yaxes(
        title_text="y_rel [m]" if col == 1 else None,
        range=[float(y.min()), float(y.max())],
        scaleanchor=f"x{axis_id}",
        scaleratio=1,
        constrain="domain",
        showgrid=False,
        zeroline=False,
        row=row,
        col=col,
    )

# ============================================================
# SELETTORI INDIPENDENTI: CAMPO E METRICA
# ============================================================

fig.update_layout(
    template="plotly_white",
    height=950,
    margin=dict(l=65, r=100, t=110, b=55),
    showlegend=False,
    uirevision="keep-view",
    updatemenus=[
        dict(
            type="dropdown",
            x=0.0,
            y=1.12,
            xanchor="left",
            yanchor="top",
            active=0,
            buttons=[
                dict(
                    label=label,
                    method="restyle",
                    args=[updates[field]],
                )
                for label, field in [
                    ("V₀", "V0"),
                    ("V(−3) / BRT", "BRT"),
                ]
            ],
        ),
        dict(
            type="dropdown",
            x=0.23,
            y=1.12,
            xanchor="left",
            yanchor="top",
            active=0,
            buttons=[
                dict(
                    label=metric,
                    method="restyle",
                    args=[{
                        "visible": [
                            owner == metric
                            for owner in trace_metrics
                        ]
                    }],
                )
                for metric in metrics
            ],
        ),
    ],
)

fig.show(config={"displayModeBar": False, "responsive": True})

## Altra visione

In [3]:
from pathlib import Path
import json
import webbrowser

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# CONFIGURAZIONE
# ============================================================

METRICS = "euclidean, dce, ttc, eggert, rss, sff, time_eggert, euc_dce_from_v0, euc_dce_minimum, eggert_ttc_from_v0, eggert_ttc_minimum"

EGO_SPEEDS = [3.0, 6.0, 9.0]
HUMAN_SPEEDS = [3.0, 6.0, 9.0]

THETA_REL = 0.0
DELTA_E = 0.0

SCALA_COMUNE_TRA_METRICHE = False

COLORSCALE = "RdYlGn"

EGO_LENGTH = 4.68
EGO_WIDTH = 2.20


THETA_OPTIONS = [
    ("-π", -np.pi),
    ("-3π/4", -3 * np.pi / 4),
    ("-π/2", -np.pi / 2),
    ("-π/3", -np.pi / 3),
    ("-π/4", -np.pi / 4),
    ("-π/6", -np.pi / 6),
    ("-π/12", -np.pi/12),
    ("0", 0.0),
    ("π/12", np.pi/12),
    ("π/6", np.pi / 6),
    ("π/4", np.pi / 4),
    ("π/3", np.pi / 3),
    ("π/2", np.pi / 2),
    ("3π/4", 3 * np.pi / 4),
    ("π", np.pi),
]

DELTA_OPTIONS = [
    ("-π/12", -np.pi / 12),
    ("0", 0.0),
    ("π/12", np.pi / 12),
]


# ============================================================
# FUNZIONI
# ============================================================

def nearest(axis, value):
    return int(np.argmin(np.abs(axis - value)))


def nearest_periodic(axis, value):
    distance = np.abs(
        np.angle(
            np.exp(1j * (axis - value))
        )
    )
    return int(np.argmin(distance))


def color_limits(values):
    low = min(float(np.nanmin(z)) for z in values)
    high = max(float(np.nanmax(z)) for z in values)

    if low == high:
        padding = max(abs(low), 1.0) * 1e-6
        low -= padding
        high += padding

    return low, high


# ============================================================
# METRICHE E CARTELLA RISULTATI
# ============================================================

metrics = list(dict.fromkeys(
    name.strip().lower()
    for name in METRICS.split(",")
    if name.strip()
))

valid_metrics = {
    "euclidean",
    "dce",
    "ttc",
    "eggert",
    "time_eggert",
    "rss",
    "sff",
    "euc_dce_from_v0",
    "euc_dce_minimum",
    "eggert_ttc_from_v0",
    "eggert_ttc_minimum"
}

if not metrics or not set(metrics) <= valid_metrics:
    raise ValueError(
        "METRICS deve contenere metriche valide separate da virgole."
    )

if len(EGO_SPEEDS) != 3 or len(HUMAN_SPEEDS) != 3:
    raise ValueError(
        "Servono esattamente tre velocità ego e tre human."
    )

root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "results" / "brt").is_dir()
    ),
    None,
)

if root is None:
    raise FileNotFoundError(
        "Cartella results/brt non trovata."
    )


# ============================================================
# ASSI DI RIFERIMENTO
# ============================================================

first_path = (
    root
    / "results"
    / "brt"
    / f"{metrics[0]}.npz"
)

with np.load(first_path, allow_pickle=False) as data:
    reference_axes = {
        name: np.asarray(data[name], dtype=float)
        for name in [
            "x_rel",
            "y_rel",
            "theta_rel",
            "v_H",
            "delta_E",
            "v_E",
        ]
    }

x = reference_axes["x_rel"]
y = reference_axes["y_rel"]
theta_axis = reference_axes["theta_rel"]
delta_axis = reference_axes["delta_E"]

i_human = [
    nearest(reference_axes["v_H"], value)
    for value in HUMAN_SPEEDS
]

i_ego = [
    nearest(reference_axes["v_E"], value)
    for value in EGO_SPEEDS
]

if len(set(i_human)) != 3 or len(set(i_ego)) != 3:
    raise ValueError(
        "Due velocità selezionano lo stesso nodo."
    )

human_used = reference_axes["v_H"][i_human]
ego_used = reference_axes["v_E"][i_ego]


# ============================================================
# THETA E DELTA DISPONIBILI
# ============================================================

theta_selection = [
    (
        label,
        nearest_periodic(theta_axis, value),
    )
    for label, value in THETA_OPTIONS
]

delta_selection = [
    (
        label,
        nearest(delta_axis, value),
    )
    for label, value in DELTA_OPTIONS
]

theta_indices = sorted({
    index
    for _, index in theta_selection
})

delta_indices = sorted({
    index
    for _, index in delta_selection
})


# ============================================================
# VALORI INIZIALI
# ============================================================

initial_theta_label, initial_theta_idx = min(
    theta_selection,
    key=lambda item: abs(
        np.angle(
            np.exp(
                1j * (
                    theta_axis[item[1]]
                    - THETA_REL
                )
            )
        )
    ),
)

initial_delta_label, initial_delta_idx = min(
    delta_selection,
    key=lambda item: abs(
        delta_axis[item[1]]
        - DELTA_E
    ),
)

initial_metric = metrics[0]
initial_field = "V0"


# ============================================================
# CARICAMENTO DELLE SLICE
# ============================================================

loaded = {}

for metric in metrics:

    path = (
        root
        / "results"
        / "brt"
        / f"{metric}.npz"
    )

    print(f"Loading {metric}...")

    with np.load(path, allow_pickle=False) as data:

        axes = {
            name: np.asarray(data[name], dtype=float)
            for name in [
                "x_rel",
                "y_rel",
                "theta_rel",
                "v_H",
                "delta_E",
                "v_E",
            ]
        }

        for name, axis in axes.items():
            reference = reference_axes[name]

            if (
                axis.shape != reference.shape
                or not np.allclose(axis, reference)
            ):
                raise ValueError(
                    f"{metric}: griglia diversa lungo {name}."
                )

        if "target_time" in data.files:
            if not np.isclose(
                float(data["target_time"]),
                -3.0,
            ):
                raise ValueError(
                    f"{metric}: il BRT non è a t = -3 s."
                )

        loaded[metric] = {
            "V0": {},
            "BRT": {},
        }

        for field in ("V0", "BRT"):

            volume = data[field]

            for i_theta in theta_indices:
                for i_delta in delta_indices:

                    key = f"{i_theta}_{i_delta}"

                    loaded[metric][field][key] = [
                        np.array(
                            volume[
                                :,
                                :,
                                i_theta,
                                ih,
                                i_delta,
                                ie,
                            ].T,
                            dtype=np.float32,
                            copy=True,
                        )
                        for ih in i_human
                        for ie in i_ego
                    ]

            del volume

print("Loading completed.")


# ============================================================
# LIMITI COLORBAR
# ============================================================

scale_limits = {}

for field in ("V0", "BRT"):

    if SCALA_COMUNE_TRA_METRICHE:

        all_values = [
            z
            for metric in metrics
            for slices in loaded[metric][field].values()
            for z in slices
        ]

        low, high = color_limits(all_values)

        for metric in metrics:
            scale_limits[
                (metric, field)
            ] = (low, high)

    else:

        for metric in metrics:

            metric_values = [
                z
                for slices in loaded[metric][field].values()
                for z in slices
            ]

            scale_limits[
                (metric, field)
            ] = color_limits(metric_values)


# ============================================================
# FIGURA INIZIALE
# ============================================================

initial_key = (
    f"{initial_theta_idx}_{initial_delta_idx}"
)

initial_values = (
    loaded[initial_metric][initial_field][initial_key]
)

initial_v0 = (
    loaded[initial_metric]["V0"][initial_key]
)

low, high = scale_limits[
    (initial_metric, initial_field)
]

fig = make_subplots(
    rows=3,
    cols=3,
    horizontal_spacing=0.055,
    vertical_spacing=0.09,
    subplot_titles=[
        f"v_H = {vh:.2f} m/s · v_E = {ve:.2f} m/s"
        for vh in human_used
        for ve in ego_used
    ],
)


# Ogni pannello contiene:
# 0) Heatmap
# 1) contorno zero del campo visualizzato
# 2) contorno V0 opzionale sul BRT

for panel in range(9):

    row, col = divmod(panel, 3)
    row += 1
    col += 1

    fig.add_trace(
        go.Heatmap(
            x=x,
            y=y,
            z=initial_values[panel],
            zmin=low,
            zmax=high,
            zsmooth="best",
            colorscale=COLORSCALE,
            showscale=(panel == 2),
            colorbar=dict(
                x=1.02,
                y=0.5,
                len=1.0,
                thickness=15,
                tickformat=".2g",
                title="V₀",
            ),
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Contour(
            x=x,
            y=y,
            z=initial_values[panel],
            autocontour=False,
            contours=dict(
                start=0,
                end=0,
                size=1,
                coloring="none",
                showlabels=False,
            ),
            line=dict(
                color="black",
                width=2.5,
            ),
            showscale=False,
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Contour(
            x=x,
            y=y,
            z=initial_v0[panel],
            autocontour=False,
            contours=dict(
                start=0,
                end=0,
                size=1,
                coloring="none",
                showlabels=False,
            ),
            line=dict(
                color="deepskyblue",
                width=2.5,
                dash="dash",
            ),
            showscale=False,
            hoverinfo="skip",
            visible=False,
        ),
        row=row,
        col=col,
    )


# ============================================================
# EGO E ASSI
# ============================================================

for panel in range(9):

    row, col = divmod(panel, 3)
    row += 1
    col += 1

    axis_id = "" if panel == 0 else str(panel + 1)

    for color, width in [
        ("white", 5),
        ("black", 2),
    ]:
        fig.add_shape(
            type="rect",
            x0=-EGO_LENGTH / 2,
            x1=EGO_LENGTH / 2,
            y0=-EGO_WIDTH / 2,
            y1=EGO_WIDTH / 2,
            line=dict(
                color=color,
                width=width,
            ),
            fillcolor="rgba(0,0,0,0)",
            layer="above",
            row=row,
            col=col,
        )

    fig.update_xaxes(
        title_text=(
            "x_rel [m]"
            if row == 3
            else None
        ),
        range=[
            float(x.min()),
            float(x.max()),
        ],
        constrain="domain",
        showgrid=False,
        zeroline=False,
        row=row,
        col=col,
    )

    fig.update_yaxes(
        title_text=(
            "y_rel [m]"
            if col == 1
            else None
        ),
        range=[
            float(y.min()),
            float(y.max()),
        ],
        scaleanchor=f"x{axis_id}",
        scaleratio=1,
        constrain="domain",
        showgrid=False,
        zeroline=False,
        row=row,
        col=col,
    )


fig.update_layout(
    template="plotly_white",
    height=950,
    margin=dict(
        l=65,
        r=100,
        t=120,
        b=55,
    ),
    showlegend=False,
    uirevision="keep-view",
    title=dict(
        text=(
            f"{initial_metric.upper()} — V₀"
            f"<br>"
            f"θ_rel = {initial_theta_label} · "
            f"δ_E = {initial_delta_label}"
            f"<br>"
        ),
        x=0.5,
        xanchor="center",
    ),
)


# ============================================================
# DATI DA PASSARE A JAVASCRIPT
# ============================================================

payload = {}

for metric in metrics:

    payload[metric] = {
        "V0": {},
        "BRT": {},
        "limits": {},
    }

    for field in ("V0", "BRT"):

        payload[metric]["limits"][field] = list(
            scale_limits[(metric, field)]
        )

        for key, slices in loaded[metric][field].items():

            payload[metric][field][key] = [
                z.tolist()
                for z in slices
            ]


theta_js = [
    {
        "label": label,
        "index": int(index),
    }
    for label, index in theta_selection
]

delta_js = [
    {
        "label": label,
        "index": int(index),
    }
    for label, index in delta_selection
]


# ============================================================
# HTML STANDALONE
# ============================================================

plot_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    div_id="brtFigure",
    config={
        "displayModeBar": False,
        "responsive": True,
    },
)

controls_html = f"""
<div id="controls">
    <div class="control">
        <label for="fieldSelect">Campo</label>
        <select id="fieldSelect">
            <option value="V0">V₀</option>
            <option value="BRT">V(−3) / BRT</option>
        </select>
    </div>

    <div class="control">
        <label for="metricSelect">Metrica</label>
        <select id="metricSelect">
            {
                ''.join(
                    f'<option value="{metric}">{metric}</option>'
                    for metric in metrics
                )
            }
        </select>
    </div>

    <div class="control">
        <label for="thetaSelect">θ_rel</label>
        <select id="thetaSelect">
            {
                ''.join(
                    f'<option value="{item["index"]}" '
                    f'data-label="{item["label"]}">'
                    f'{item["label"]}</option>'
                    for item in theta_js
                )
            }
        </select>
    </div>

    <div class="control">
        <label for="deltaSelect">δ_E</label>
        <select id="deltaSelect">
            {
                ''.join(
                    f'<option value="{item["index"]}" '
                    f'data-label="{item["label"]}">'
                    f'{item["label"]}</option>'
                    for item in delta_js
                )
            }
        </select>
    </div>

    <div class="control toggle-control">
        <label for="v0Overlay">V₀ = 0 sul BRT</label>
        <label class="switch">
            <input
                type="checkbox"
                id="v0Overlay"
            >
            <span class="slider"></span>
        </label>
    </div>
</div>
"""


javascript = f"""
<script>

const DATA = {json.dumps(payload)};
const THETA_OPTIONS = {json.dumps(theta_js)};
const DELTA_OPTIONS = {json.dumps(delta_js)};

const gd = document.getElementById("brtFigure");

const fieldSelect =
    document.getElementById("fieldSelect");

const metricSelect =
    document.getElementById("metricSelect");

const thetaSelect =
    document.getElementById("thetaSelect");

const deltaSelect =
    document.getElementById("deltaSelect");

const v0Overlay =
    document.getElementById("v0Overlay");


const heatIndices = [
    0, 3, 6,
    9, 12, 15,
    18, 21, 24
];

const fieldContourIndices = [
    1, 4, 7,
    10, 13, 16,
    19, 22, 25
];

const v0ContourIndices = [
    2, 5, 8,
    11, 14, 17,
    20, 23, 26
];


function selectedLabel(select) {{
    return select
        .options[select.selectedIndex]
        .dataset.label;
}}


function updateFigure() {{

    const metric = metricSelect.value;
    const field = fieldSelect.value;

    const thetaIndex =
        thetaSelect.value;

    const deltaIndex =
        deltaSelect.value;

    const thetaLabel =
        selectedLabel(thetaSelect);

    const deltaLabel =
        selectedLabel(deltaSelect);

    const key =
        thetaIndex + "_" + deltaIndex;

    const values =
        DATA[metric][field][key];

    const v0Values =
        DATA[metric]["V0"][key];

    const limits =
        DATA[metric]["limits"][field];

    const low = limits[0];
    const high = limits[1];

    const showV0 =
        field === "BRT"
        && v0Overlay.checked;


    Plotly.restyle(
        gd,
        {{
            z: values,
            zmin: new Array(9).fill(low),
            zmax: new Array(9).fill(high)
        }},
        heatIndices
    );


    Plotly.restyle(
        gd,
        {{
            z: values
        }},
        fieldContourIndices
    );


    Plotly.restyle(
        gd,
        {{
            z: v0Values,
            visible:
                new Array(9).fill(showV0)
        }},
        v0ContourIndices
    );


    Plotly.restyle(
        gd,
        {{
            "colorbar.title.text":
                field === "V0"
                ? "V₀"
                : "V(−3)"
        }},
        [6]
    );


    const fieldText =
        field === "V0"
        ? "V₀"
        : "BRT V(−3)";


    if (showV0) {{
        contourText +=
            " · azzurro tratteggiato: V₀ = 0";
    }}


    Plotly.relayout(
        gd,
        {{
            "title.text":
                metric.toUpperCase()
                + " — "
                + fieldText
                + "<br>"
                + "θ_rel = "
                + thetaLabel
                + " · δ_E = "
                + deltaLabel
                + "<br>"
                + contourText
        }}
    );


    v0Overlay.disabled =
        field !== "BRT";

    document
        .querySelector(".toggle-control")
        .classList.toggle(
            "disabled",
            field !== "BRT"
        );
}}


fieldSelect.addEventListener(
    "change",
    updateFigure
);

metricSelect.addEventListener(
    "change",
    updateFigure
);

thetaSelect.addEventListener(
    "change",
    updateFigure
);

deltaSelect.addEventListener(
    "change",
    updateFigure
);

v0Overlay.addEventListener(
    "change",
    updateFigure
);


fieldSelect.value =
    "{initial_field}";

metricSelect.value =
    "{initial_metric}";

thetaSelect.value =
    "{initial_theta_idx}";

deltaSelect.value =
    "{initial_delta_idx}";

updateFigure();

</script>
"""


css = """
<style>

body {
    margin: 0;
    padding: 20px;
    font-family:
        Arial,
        Helvetica,
        sans-serif;
    background: white;
}

#controls {
    display: flex;
    flex-wrap: wrap;
    gap: 18px;
    align-items: flex-end;
    padding:
        14px
        18px;
    margin-bottom: 10px;
    border:
        1px solid
        #dddddd;
    border-radius: 8px;
    background: #fafafa;
}

.control {
    display: flex;
    flex-direction: column;
    gap: 5px;
}

.control label {
    font-size: 13px;
    font-weight: 600;
}

select {
    min-width: 120px;
    padding:
        7px
        10px;
    border:
        1px solid
        #bbbbbb;
    border-radius: 5px;
    background: white;
    font-size: 14px;
}

.toggle-control {
    min-width: 140px;
}

.toggle-control.disabled {
    opacity: 0.45;
}

.switch {
    position: relative;
    display: inline-block;
    width: 46px;
    height: 24px;
}

.switch input {
    opacity: 0;
    width: 0;
    height: 0;
}

.slider {
    position: absolute;
    cursor: pointer;
    inset: 0;
    background-color: #cccccc;
    transition: 0.2s;
    border-radius: 24px;
}

.slider:before {
    position: absolute;
    content: "";
    height: 18px;
    width: 18px;
    left: 3px;
    bottom: 3px;
    background-color: white;
    transition: 0.2s;
    border-radius: 50%;
}

input:checked + .slider {
    background-color: #444444;
}

input:checked + .slider:before {
    transform:
        translateX(22px);
}

#brtFigure {
    width: 100%;
}

</style>
"""


html = f"""
<!DOCTYPE html>

<html>

<head>
    <meta charset="utf-8">
    <title>BRT 3x3 interactive</title>
    {css}
</head>

<body>

    {controls_html}

    {plot_html}

    {javascript}

</body>

</html>
"""


# ============================================================
# SALVATAGGIO E APERTURA
# ============================================================

output_path = (
    root
    / "examples"
    / "brt_3x3_interactive.html"
)

output_path.write_text(
    html,
    encoding="utf-8",
)

print()
print("HTML saved in:")
print(output_path.resolve())

webbrowser.open(
    output_path.resolve().as_uri()
)

Loading euclidean...
Loading dce...
Loading ttc...
Loading eggert...
Loading rss...
Loading sff...
Loading time_eggert...
Loading euc_dce_from_v0...
Loading euc_dce_minimum...
Loading eggert_ttc_from_v0...
Loading eggert_ttc_minimum...
Loading completed.

HTML saved in:
/home/alessio-grisorio/TESI/hj_reachability/examples/brt_3x3_interactive.html


True

Gtk-Message: 17:27:10.142: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
